# MMIS 671 Final Exam: Classification using *TensorFlow* and *Keras*
You must develop a neural network model to classify 28×28 grayscale images of items using the Keras Fashion MNIST dataset (https://keras.io/api/datasets/fashion_mnist/). The dataset consists of 60,000 examples for training and 10,000 examples for testing. Each example is a labeled image of an item. Then use the trained model to predict the class for 40 unlabeled examples. Use TensorFlow and Keras models for this task.

## Import libraries

In [15]:
import numpy as np # for computation
import time
import pandas as pd # for data handling and analysis
import matplotlib.pyplot as plt # for plotting

# metrics to evaluate models
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import plot_roc_curve

# import tensorFlow / keras for neural networks
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical # for one-hot representation
from tensorflow.keras.layers import Dense, Conv2D # dense and convolution layer
from tensorflow.keras.layers import Dropout, BatchNormalization # for regularization
from tensorflow.keras.layers import Flatten # to flatten convolution layer output
from tensorflow.keras.layers import MaxPooling2D # for max pooling
from tensorflow.keras.layers import Activation # for activation functions 
from tensorflow.keras.optimizers import SGD, Adam # for optimization
from tensorflow.keras.utils import plot_model # to display model
from tensorflow.keras.callbacks import ModelCheckpoint # to save best model
from tensorflow.keras.models import load_model # to load saved model

from keras.datasets import fashion_mnist # to import data for training and testing 

from google.colab import drive 
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Task 0: Get data
Read *MNIST* labeled image data for training and testing classifiers into *Numpy* arrays. Check the shape of the arrays and the distribution of digits in the training and test data.

In [16]:
# Read MNIST labeled image data for training and testing
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data() 

# Check shape of arrays
print('%d training and %d test examples read' %(len(y_train), len(y_test)))
print("Shape of arrays:")
print("\tTraining -> input x_train: %s, output y_train: %s" %(x_train.shape, y_train.shape))
print("\tTest -> input x_test: %s, output y_test: %s" %(x_test.shape, y_test.shape))
print()

# Check class distributions
print("Distribution of digits in training data:")
digits, counts = np.unique(y_train, return_counts=True)
print("Counts:")
print(pd.DataFrame(counts).T)
print("Proportions:")
print(pd.DataFrame((counts/counts.sum()).round(2)).T)
print()

print("Distribution of digits in test data:")
digits, counts = np.unique(y_test, return_counts=True)
print("Counts:")
print(pd.DataFrame(counts).T)
print("Proportions:")
print(pd.DataFrame((counts/counts.sum()).round(2)).T)

60000 training and 10000 test examples read
Shape of arrays:
	Training -> input x_train: (60000, 28, 28), output y_train: (60000,)
	Test -> input x_test: (10000, 28, 28), output y_test: (10000,)

Distribution of digits in training data:
Counts:
      0     1     2     3     4     5     6     7     8     9
0  6000  6000  6000  6000  6000  6000  6000  6000  6000  6000
Proportions:
     0    1    2    3    4    5    6    7    8    9
0  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1

Distribution of digits in test data:
Counts:
      0     1     2     3     4     5     6     7     8     9
0  1000  1000  1000  1000  1000  1000  1000  1000  1000  1000
Proportions:
     0    1    2    3    4    5    6    7    8    9
0  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1  0.1


Read unlabeled image data from "*unlabeled_items.npy*" into *Numpy* array and check the shape of the array.

In [17]:
# Read unlabeled data
x_unlabeled = np.load('/content/drive/My Drive/Colab Notebooks/unlabeled_items.npy')
print("Shape of x_unlabeled: ", x_unlabeled.shape)

Shape of x_unlabeled:  (40, 28, 28)


## Task 1b: Train and test convolution neural network 

**Function to prepare inputs for CNN**

Since a Convolution Neural Network (CNN) needs 3D arrays as inputs, we shall convert the 2D arrays x_train and x_test into 3D arrays. We shall also normalize the data by mapping gray scale values (0 – 255) to a number between 0 and 1.

In [18]:
def Xform_cnn(x):
    return x.reshape(-1, 28, 28, 1)/255.0

**Function to create CNN**

The function ***CNN_4***(*W, H, nClasses, K1, K2, K3, K4, n, dropout_rate, kernel_size, pool_size*) returns a CNN with 4 convolution layers followed by a densely connected layer and an output layer with specified parameters, where:
- *W, H*: are the image width and height
- *nClasses*: is the number of output classes
- K1, K2, K3, K4: are the number of kernels (filters) in the 4 convolution layers
- *n*: in the number of neurons in the densely connected layer
- *dropout_rate*: specifies the dropout rate
- *kernel_size*: specifies the shape of the kernels
- *pool_size*: specifies the shape for max-pooling

You can define similar functions to create CNN models with other architectures if you like

In [19]:
def cnn4(W, H, nClasses,
         K1, K2, K3, K4, n,
         dropOutRate, 
         kernel_size, 
         pool_size):
  """network with 4 convolution layers and a dense layer
  K1, K2, K3, K4: number of filters in the 4 convoluttion layers
  n: number of neurons in dense layer 
  """ 
  model = Sequential()
  
  # Covolution layers
  model.add(Conv2D(K1, kernel_size, activation='relu', 
                  input_shape=(W, H, 1)))
  model.add(Conv2D(K2, kernel_size, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPooling2D(pool_size))
  model.add(Dropout(dropOutRate))
  
  model.add(Conv2D(K3, kernel_size, activation='relu')) 
  model.add(Conv2D(K4, kernel_size, activation='relu'))
  model.add(BatchNormalization())
  model.add(MaxPooling2D(pool_size))
  model.add(Dropout(dropOutRate))

  # Dense layer
  model.add(Flatten())
  model.add(Dense(n, activation='relu'))
  model.add(BatchNormalization())
  model.add(Dropout(dropOutRate))
  
  # Output layer
  model.add(Dense(nClasses, activation='softmax'))
  # compile model
  opt = SGD(lr=0.01, momentum=0.9)
  model.compile(optimizer=opt, loss='categorical_crossentropy', 
                metrics=['accuracy'])
  
  return model

**Specify model parameters**

Specify parameters and use function *cnn4* to create a convolution neural network model.

In [20]:
# Specify parameters for model (change as needed)
W, H = 28, 28 # image shape
nClasses = 10 # number of classes
K1, K2, K3, K4, n = 16, 32, 64, 128, 256
dropOutRate = 0.2
kernel_size = (3,3) 
pool_size = (2,2)

# Specify parameters for training (change as needed)
epochs = 20 # number of training epochs
batch_size = 256 # batch_size for training 

print("1. Number of convolution layers: %d" %len([K1, K2, K3, K4]))
print("2. Number of filters in the convolution layers:", tuple([K1, K2, K3, K4]))
print("3. Width (and height) of filters F = ", kernel_size[0])
print("4. Number of dense layers following the convolution layer (excluding the output layer) = %d" %len([n]))
print("5. Dense layer sizes:", n)
print("6. Dropout rate used = %4.2f" %dropout_rate)
print("7. Number of training epochs = %d" %epochs)
print("8. Batch size used = %d" %batch_size)

# create model
cnn = cnn4(W, H, nClasses,K1, K2, K3, K4, n, dropOutRate, kernel_size, pool_size)
print("9. Model summary:")
print(cnn.summary())
print("Plot model:")
plot_model(cnn, to_file='cnn.png') # save and display architecture

1. Number of convolution layers: 4
2. Number of filters in the convolution layers: (16, 32, 64, 128)
3. Width (and height) of filters F =  3
4. Number of dense layers following the convolution layer (excluding the output layer) = 1
5. Dense layer sizes: 256


NameError: ignored

How many trainable weights (including bias weights) does your model contain? 

We shall compute it from the model summary by only taking into account the parameters associated with convolution layers and dense layers.

In [12]:
n_wts = 160 + 4640 + 18496 + 73856 + 524544 + 2570 # change as needed based on your model
print("10.	Model contains %d trainable weights" %n_wts)

10.	Model contains 624266 trainable weights


Train model

In [13]:
history = cnn.fit(Xform_cnn(x_train), to_categorical(y_train),
                    epochs=epochs, batch_size=batch_size, validation_split=0.1)

# plot history
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()

NameError: ignored

## Task 2. Evaluate trained cnn

In [21]:
%%time
pred_cnn = cnn.predict(Xform_cnn(x_test)).argmax(axis=1) # predicted classes
acc_cnn = accuracy_score(y_test, pred_cnn) # accuracy

print("11.	Accuracy for the 10,000 test examples = %4.4f" %acc_cnn)

print("12.	Classification report for the 10,000 test examples:")
print(classification_report(y_test, pred_cnn, digits=4))

print("13.	Confusion matrix for the 10,000 test examples::")
cm_cnn = pd.DataFrame(confusion_matrix(y_test, pred_cnn))
cm_cnn.to_csv("confusion_matrix_cnn.csv") # save confusion_matrix
cm_cnn # display confusion_matrix

NameError: ignored

## Task 3. Predidict digit for unlabeled examples

In [ ]:
# predict using cnn
pred_new = cnn.predict(Xform_cnn(x_unlabeled)).argmax(axis=1) # predicted classes
pred_new # show predictions 

We shall display the unlabeled examples with predicted labels

In [ ]:
def displayDigits(images, labels, nCols=10):
    """Displays images with labels (nCols per row)
    - images: list of vectors with 784 (28x28) grayscale values
    - labels: list of labels for images"""
    nRows = np.ceil(len(labels)/nCols).astype('int') # number of rows
    plt.figure(figsize=(2*nCols,2*nRows)) # figure size
    for i in range(len(labels)):
        plt.subplot(nRows,nCols,i+1)
        plt.xticks([])
        plt.yticks([])
        plt.grid(False)
        plt.imshow(images[i], interpolation='nearest')
        plt.xlabel(str(labels[i]), fontsize=14)
    plt.show()
    return

displayDigits(x_unlabeled, pred_new, nCols=10)